# Transformer utilities

> Typed encoder–decoder transformer building blocks used by Chapter 2.

The implementation is adapted from Mark Liu’s `txt2img` repository under the MIT License, then organized, typed, and documented for reuse in this companion project.


In [ ]:
#| default_exp utils.transformer


## Reusable implementation

The public interface keeps the upstream teaching code available while using descriptive names, explicit tensor shapes, and compatibility aliases where needed.


In [ ]:
#| export
import math
from collections.abc import Callable
from copy import deepcopy
from typing import Protocol

import torch
from numpy.typing import NDArray
from torch import Tensor, nn

DEFAULT_DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Backward-compatible name used by the book's original notebooks.
DEVICE: str = DEFAULT_DEVICE.type


class OptimizerSchedule(Protocol):
    """Interface required by :class:`SimpleLossCompute`."""

    optimizer: torch.optim.Optimizer

    def step(self) -> None:
        """Advance the wrapped optimizer by one step."""


def subsequent_mask(size: int) -> Tensor:
    """Create a mask that hides tokens after the current position.

    Args:
        size: Sequence length.

    Returns:
        A Boolean tensor shaped ``(1, size, size)``. ``True`` entries are
        visible to attention.
    """
    future_positions: Tensor = torch.triu(torch.ones((1, size, size), dtype=torch.bool), diagonal=1)
    return ~future_positions


def make_std_mask(target: Tensor, padding_id: int) -> Tensor:
    """Combine target padding and causal-attention masks.

    Args:
        target: Token IDs shaped ``(batch_size, sequence_length)``.
        padding_id: Token ID used for padding.

    Returns:
        A Boolean mask shaped
        ``(batch_size, sequence_length, sequence_length)``.
    """
    padding_mask: Tensor = (target != padding_id).unsqueeze(-2)
    causal_mask: Tensor = subsequent_mask(target.size(-1)).to(target.device)
    return padding_mask & causal_mask.type_as(padding_mask)


class Batch:
    """Prepare source and shifted target tensors for transformer training.

    Args:
        src: NumPy source-token IDs shaped
            ``(batch_size, source_sequence_length)``.
        trg: Optional NumPy target-token IDs shaped
            ``(batch_size, target_sequence_length)``.
        pad: Token ID used for padding.
        device: Device on which tensors are created.
    """

    def __init__(
        self,
        src: NDArray,
        trg: NDArray | None = None,
        pad: int = 0,
        device: torch.device | str = DEFAULT_DEVICE,
    ) -> None:
        source: Tensor = torch.from_numpy(src).to(device=device, dtype=torch.long)
        self.src: Tensor = source
        self.src_mask: Tensor = (source != pad).unsqueeze(-2)

        self.trg: Tensor | None = None
        self.trg_y: Tensor | None = None
        self.trg_mask: Tensor | None = None
        self.ntokens: Tensor | None = None
        if trg is not None:
            target: Tensor = torch.from_numpy(trg).to(device=device, dtype=torch.long)
            self.trg = target[:, :-1]
            self.trg_y = target[:, 1:]
            self.trg_mask = make_std_mask(self.trg, pad)
            self.ntokens = (self.trg_y != pad).sum()


class Transformer(nn.Module):
    """Encoder-decoder transformer with source and target embeddings."""

    def __init__(
        self,
        encoder: nn.Module,
        decoder: nn.Module,
        src_embed: nn.Module,
        tgt_embed: nn.Module,
        generator: nn.Module,
    ) -> None:
        """Initialize the transformer modules.

        Args:
            encoder: Source-sequence encoder.
            decoder: Target-sequence decoder.
            src_embed: Source-token embedding module.
            tgt_embed: Target-token embedding module.
            generator: Output projection module.
        """
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def encode(self, src: Tensor, src_mask: Tensor) -> Tensor:
        """Encode source token IDs into contextual representations."""
        return self.encoder(self.src_embed(src), src_mask)

    def decode(
        self,
        memory: Tensor,
        src_mask: Tensor,
        tgt: Tensor,
        tgt_mask: Tensor,
    ) -> Tensor:
        """Decode target token IDs while attending to encoded source memory."""
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

    def forward(
        self,
        src: Tensor,
        tgt: Tensor,
        src_mask: Tensor,
        tgt_mask: Tensor,
    ) -> Tensor:
        """Encode a source sequence and decode a target sequence."""
        memory: Tensor = self.encode(src, src_mask)
        return self.decode(memory, src_mask, tgt, tgt_mask)


class Encoder(nn.Module):
    """Stack identical encoder layers followed by layer normalization."""

    def __init__(self, layer: "EncoderLayer", num_layers: int) -> None:
        """Initialize the encoder stack.

        Args:
            layer: Encoder layer to copy.
            num_layers: Number of copied layers.
        """
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(layer) for _ in range(num_layers)])
        self.norm = LayerNorm(layer.size)

    def forward(self, inputs: Tensor, mask: Tensor) -> Tensor:
        """Pass embedded tokens through every encoder layer."""
        encoded: Tensor = inputs
        for layer in self.layers:
            encoded = layer(encoded, mask)
        return self.norm(encoded)


class EncoderLayer(nn.Module):
    """Self-attention and feed-forward sublayers for one encoder layer."""

    def __init__(
        self,
        size: int,
        self_attn: nn.Module,
        feed_forward: nn.Module,
        dropout: float,
    ) -> None:
        """Initialize an encoder layer."""
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([SublayerConnection(size, dropout) for _ in range(2)])
        self.size = size

    def forward(self, inputs: Tensor, mask: Tensor) -> Tensor:
        """Apply self-attention followed by the feed-forward module."""
        attended: Tensor = self.sublayer[0](
            inputs, lambda normalized: self.self_attn(normalized, normalized, normalized, mask)
        )
        return self.sublayer[1](attended, self.feed_forward)


class SublayerConnection(nn.Module):
    """Pre-normalization residual connection around a sublayer."""

    def __init__(self, size: int, dropout: float) -> None:
        """Initialize normalization and dropout."""
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs: Tensor, sublayer: Callable[[Tensor], Tensor]) -> Tensor:
        """Normalize, transform, drop out, and add the residual input."""
        return inputs + self.dropout(sublayer(self.norm(inputs)))


class LayerNorm(nn.Module):
    """Learned layer normalization used by the original implementation."""

    def __init__(self, features: int, eps: float = 1e-6) -> None:
        """Initialize scale and bias parameters."""
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, inputs: Tensor) -> Tensor:
        """Normalize the final tensor dimension."""
        mean: Tensor = inputs.mean(-1, keepdim=True)
        std: Tensor = inputs.std(-1, keepdim=True)
        z_score: Tensor = (inputs - mean) / torch.sqrt(std**2 + self.eps)
        return self.a_2 * z_score + self.b_2


class Decoder(nn.Module):
    """Stack identical decoder layers followed by layer normalization."""

    def __init__(self, layer: "DecoderLayer", num_layers: int) -> None:
        """Initialize the decoder stack."""
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(layer) for _ in range(num_layers)])
        self.norm = LayerNorm(layer.size)

    def forward(
        self,
        inputs: Tensor,
        memory: Tensor,
        src_mask: Tensor,
        tgt_mask: Tensor,
    ) -> Tensor:
        """Pass target embeddings through every decoder layer."""
        decoded: Tensor = inputs
        for layer in self.layers:
            decoded = layer(decoded, memory, src_mask, tgt_mask)
        return self.norm(decoded)


class DecoderLayer(nn.Module):
    """Masked self-attention, cross-attention, and feed-forward sublayers."""

    def __init__(
        self,
        size: int,
        self_attn: nn.Module,
        src_attn: nn.Module,
        feed_forward: nn.Module,
        dropout: float,
    ) -> None:
        """Initialize a decoder layer."""
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([SublayerConnection(size, dropout) for _ in range(3)])

    def forward(
        self,
        inputs: Tensor,
        memory: Tensor,
        src_mask: Tensor,
        tgt_mask: Tensor,
    ) -> Tensor:
        """Apply masked self-attention, cross-attention, and feed-forward."""
        decoded: Tensor = self.sublayer[0](
            inputs,
            lambda normalized: self.self_attn(normalized, normalized, normalized, tgt_mask),
        )
        decoded = self.sublayer[1](
            decoded,
            lambda normalized: self.src_attn(normalized, memory, memory, src_mask),
        )
        return self.sublayer[2](decoded, self.feed_forward)


class Embeddings(nn.Module):
    """Token embedding scaled by the square root of its dimension."""

    def __init__(self, d_model: int, vocab: int) -> None:
        """Initialize the embedding table."""
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, token_ids: Tensor) -> Tensor:
        """Embed token IDs shaped ``(batch_size, sequence_length)``."""
        return self.lut(token_ids) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    """Add fixed sinusoidal position vectors to token embeddings."""

    def __init__(
        self,
        d_model: int,
        dropout: float,
        max_len: int = 5_000,
        device: torch.device | str = DEFAULT_DEVICE,
    ) -> None:
        """Precompute sinusoidal position vectors."""
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Shape: (max_len, embedding_dim)
        positions: Tensor = torch.arange(0, max_len, dtype=torch.float32, device=device).unsqueeze(
            1
        )
        frequencies: Tensor = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32, device=device)
            * -(math.log(10_000.0) / d_model)
        )
        positional_encoding: Tensor = torch.zeros(max_len, d_model, device=device)
        phase: Tensor = positions * frequencies
        positional_encoding[:, 0::2] = torch.sin(phase)
        positional_encoding[:, 1::2] = torch.cos(phase)
        self.register_buffer("pe", positional_encoding.unsqueeze(0))

    def forward(self, inputs: Tensor) -> Tensor:
        """Add position vectors to ``(batch, sequence, embedding)`` inputs."""
        position_vectors: Tensor = self.pe[:, : inputs.size(1)].detach()
        return self.dropout(inputs + position_vectors)


def attention(
    query: Tensor,
    key: Tensor,
    value: Tensor,
    mask: Tensor | None = None,
    dropout: nn.Module | None = None,
) -> tuple[Tensor, Tensor]:
    """Compute scaled dot-product attention.

    Args:
        query: Query vectors shaped ``(..., query_length, head_dim)``.
        key: Key vectors shaped ``(..., key_length, head_dim)``.
        value: Value vectors shaped ``(..., key_length, head_dim)``.
        mask: Optional broadcastable visibility mask.
        dropout: Optional dropout applied to attention probabilities.

    Returns:
        The attended values and attention probabilities.
    """
    head_dim: int = query.size(-1)
    scores: Tensor = query @ key.transpose(-2, -1) / math.sqrt(head_dim)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    probabilities: Tensor = nn.functional.softmax(scores, dim=-1)
    if dropout is not None:
        probabilities = dropout(probabilities)
    return probabilities @ value, probabilities


class MultiHeadedAttention(nn.Module):
    """Multi-head scaled dot-product attention."""

    def __init__(self, h: int, d_model: int, dropout: float = 0.1) -> None:
        """Initialize attention projections.

        Raises:
            ValueError: If ``d_model`` is not divisible by ``h``.
        """
        super().__init__()
        if d_model % h != 0:
            raise ValueError("d_model must be divisible by the number of heads")
        self.d_k = d_model // h
        self.h = h
        self.linears = nn.ModuleList([deepcopy(nn.Linear(d_model, d_model)) for _ in range(4)])
        self.attn: Tensor | None = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(
        self,
        query: Tensor,
        key: Tensor,
        value: Tensor,
        mask: Tensor | None = None,
    ) -> Tensor:
        """Project, attend, concatenate, and project multiple heads."""
        if mask is not None:
            mask = mask.unsqueeze(1)
        batch_size: int = query.size(0)

        # Shape after projection: (batch_size, num_heads, sequence_length, head_dim)
        projected_query, projected_key, projected_value = [
            linear(inputs).view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
            for linear, inputs in zip(self.linears[:3], (query, key, value), strict=True)
        ]
        attended, self.attn = attention(
            projected_query,
            projected_key,
            projected_value,
            mask=mask,
            dropout=self.dropout,
        )
        concatenated: Tensor = (
            attended.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)
        )
        return self.linears[-1](concatenated)


class Generator(nn.Module):
    """Project decoder states to log-probabilities over target tokens."""

    def __init__(self, d_model: int, vocab: int) -> None:
        """Initialize the vocabulary projection."""
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, inputs: Tensor) -> Tensor:
        """Return log-probabilities along the vocabulary dimension."""
        return nn.functional.log_softmax(self.proj(inputs), dim=-1)


class PositionwiseFeedForward(nn.Module):
    """Two linear transformations with dropout between them.

    This preserves the book implementation, which intentionally does not place
    an activation between the two projections.
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1) -> None:
        """Initialize the two projections."""
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs: Tensor) -> Tensor:
        """Apply both position-wise projections."""
        return self.w_2(self.dropout(self.w_1(inputs)))


def create_model(
    src_vocab: int,
    tgt_vocab: int,
    num_layers: int,
    d_model: int,
    d_ff: int,
    num_heads: int,
    dropout: float = 0.1,
    device: torch.device | str = DEFAULT_DEVICE,
) -> Transformer:
    """Construct and initialize an encoder-decoder transformer.

    Args:
        src_vocab: Number of source vocabulary entries.
        tgt_vocab: Number of target vocabulary entries.
        num_layers: Number of encoder and decoder layers.
        d_model: Embedding and hidden-state size.
        d_ff: Feed-forward hidden size.
        num_heads: Number of attention heads.
        dropout: Dropout probability.
        device: Device on which to construct the model.

    Returns:
        An initialized transformer.
    """
    shared_attention = MultiHeadedAttention(num_heads, d_model).to(device)
    feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout).to(device)
    position = PositionalEncoding(d_model, dropout, device=device).to(device)

    model = Transformer(
        Encoder(
            EncoderLayer(
                d_model,
                deepcopy(shared_attention),
                deepcopy(feed_forward),
                dropout,
            ),
            num_layers,
        ),
        Decoder(
            DecoderLayer(
                d_model,
                deepcopy(shared_attention),
                deepcopy(shared_attention),
                deepcopy(feed_forward),
                dropout,
            ),
            num_layers,
        ),
        nn.Sequential(Embeddings(d_model, src_vocab), deepcopy(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), deepcopy(position)),
        Generator(d_model, tgt_vocab),
    ).to(device)

    for parameter in model.parameters():
        if parameter.dim() > 1:
            nn.init.xavier_uniform_(parameter)
    return model


class LabelSmoothing(nn.Module):
    """KL-divergence loss against a smoothed target distribution."""

    def __init__(self, size: int, padding_idx: int, smoothing: float = 0.1) -> None:
        """Initialize label smoothing."""
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist: Tensor | None = None

    def forward(self, inputs: Tensor, target: Tensor) -> Tensor:
        """Calculate loss for flattened predictions and target token IDs."""
        if inputs.size(1) != self.size:
            raise ValueError("prediction vocabulary dimension does not match size")
        true_distribution: Tensor = inputs.detach().clone()
        true_distribution.fill_(self.smoothing / (self.size - 2))
        true_distribution.scatter_(1, target.detach().unsqueeze(1), self.confidence)
        true_distribution[:, self.padding_idx] = 0
        padding_rows: Tensor = torch.nonzero(
            target.detach() == self.padding_idx, as_tuple=False
        ).squeeze(-1)
        if padding_rows.numel() > 0:
            true_distribution.index_fill_(0, padding_rows, 0.0)
        self.true_dist = true_distribution
        return self.criterion(inputs, true_distribution.clone())


class SimpleLossCompute:
    """Calculate normalized loss and optionally update model parameters."""

    def __init__(
        self,
        generator: nn.Module,
        criterion: nn.Module,
        opt: OptimizerSchedule | None = None,
    ) -> None:
        """Initialize the loss helper."""
        self.generator = generator
        self.criterion = criterion
        self.opt = opt

    def __call__(self, x: Tensor, y: Tensor, norm: Tensor) -> Tensor:
        """Calculate loss, backpropagate, and optionally take an optimizer step."""
        log_probabilities: Tensor = self.generator(x)
        loss: Tensor = (
            self.criterion(
                log_probabilities.contiguous().view(-1, log_probabilities.size(-1)),
                y.contiguous().view(-1),
            )
            / norm
        )
        loss.backward()
        if self.opt is not None:
            self.opt.step()
            self.opt.optimizer.zero_grad()
        return loss.detach().item() * norm.float()


class NoamOpt:
    """Optimizer wrapper implementing the transformer warmup schedule."""

    def __init__(
        self,
        model_size: int,
        factor: float,
        warmup: int,
        optimizer: torch.optim.Optimizer,
    ) -> None:
        """Initialize the schedule and wrapped optimizer."""
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size
        self._rate = 0.0

    def step(self) -> None:
        """Update the learning rate and take one optimizer step."""
        self._step += 1
        learning_rate: float = self.rate()
        for parameter_group in self.optimizer.param_groups:
            parameter_group["lr"] = learning_rate
        self._rate = learning_rate
        self.optimizer.step()

    def rate(self, step: int | None = None) -> float:
        """Return the scheduled learning rate for a step."""
        current_step: int = self._step if step is None else step
        return self.factor * (
            self.model_size ** (-0.5)
            * min(
                current_step ** (-0.5),
                current_step * self.warmup ** (-1.5),
            )
        )

## Summary

This module provides reusable, documented model blocks without depending on notebook execution state.
